# COLOC ERA5

In [82]:
import numpy as np
import pandas as pd
import xarray as xr
import dask.dataframe as dd


import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250m, browse_swot_2km,  add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature


import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

In [88]:
drifters_sources

'all_med_variational_10min_v1.nc'

In [91]:
os.path.join(zarr_dir,'coloc_files', 'wind', f'windcoloc_spectral_decomp_{dt}_'+drifters_sources.replace(".nc", ".csv"))

'/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/windcoloc_spectral_decomp_12h_all_med_variational_10min_v1.csv'

In [2]:
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster
    from dask import config

    config.set({"distributed.comm.timeouts.connect": "200s"})
    cluster = PBSCluster(cores=5, processes=5, walltime="01:00:00")
    # cluster = PBSCluster(cores=20, processes=20, walltime='02:00:00')#8
    w = cluster.scale(jobs=2)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.scheduler.transition-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.comm.recent-messages-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  w

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.0.188:8787/status,
Dashboard: http://10.148.0.188:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.0.188:47923,Workers: 0
Dashboard: http://10.148.0.188:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [10]:
cluster.close()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

# CHOOSE

In [86]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v1.nc'

# Drifters param
dt = '12h'
drifter_preprocess = 'spectral_decomp' # '', 'spectral_decomp', 'low_pass'

if drifter_preprocess == True : drifters_sources = 'spectral_decomp_'+ drifters_sources

colocs_source = f'{dt}_{drifter_preprocess}_{drifters_sources}'

# Drifters
ddf = dd.read_csv(os.path.join(zarr_dir, 'coloc_files', 'drifters', f'drifterscoloc_'+colocs_source.replace('.nc', '.csv')), dtype={'drifter_id':str}, parse_dates=['datetime']).set_index('row_number').repartition(npartitions=10)

In [29]:
df = ddf.compute()

In [10]:
df

,pass_number,drifter_id,datetime,x,y,cruise_id,lonc,latc,longitude,latitude,...,semidiurnal_velocity_east,semidiurnal_velocity_north,semidiurnal_acceleration_east,semidiurnal_acceleration_north,inside_left,inside_right,drifter_type,time_to_swot,cycle_number,cycle_date
row_number,,,,,,,,,,,,,,,,,,,,,
0,3,0-4388554,2023-03-28 12:30:00,-60562.982529,108784.805685,C-SWOT,6.413977,41.954212,5.672100,42.931158,...,-0.000554,-0.002285,-7.387893e-07,1.972846e-06,0.0,1.0,CARTHE,0 days 11:50:02.155782592,474,2023-03-29 00:20:02.155782592
1,3,0-4388554,2023-03-28 12:40:00,-60688.465110,108825.137036,C-SWOT,6.413977,41.954212,5.670559,42.931511,...,-0.000996,-0.001092,-7.310052e-07,1.996478e-06,0.0,1.0,CARTHE,0 days 11:40:02.155782592,474,2023-03-29 00:20:02.155782592
2,3,0-4388554,2023-03-28 12:50:00,-60814.306712,108866.052832,C-SWOT,6.413977,41.954212,5.669013,42.931870,...,-0.001432,0.000110,-7.174299e-07,2.004784e-06,0.0,1.0,CARTHE,0 days 11:30:02.155782592,474,2023-03-29 00:20:02.155782592
3,3,0-4388554,2023-03-28 13:00:00,-60940.478999,108907.580544,C-SWOT,6.413977,41.954212,5.667463,42.932234,...,-0.001857,0.001313,-6.981558e-07,1.997684e-06,0.0,1.0,CARTHE,0 days 11:20:02.155782592,474,2023-03-29 00:20:02.155782592
4,3,0-4388554,2023-03-28 13:10:00,-61066.927804,108949.746682,C-SWOT,6.413977,41.954212,5.665910,42.932603,...,-0.002269,0.002508,-6.733199e-07,1.975223e-06,0.0,1.0,CARTHE,0 days 11:10:02.155782592,474,2023-03-29 00:20:02.155782592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377480,16,300534060218400,2023-07-10 03:10:00,-273201.488915,-13922.549719,C-SWOT,4.825800,38.549400,1.698423,38.382160,...,0.020580,-0.025553,-8.536752e-07,-1.354456e-06,0.0,1.0,SVP,0 days 08:13:38.131544384,577,2023-07-09 18:56:21.868455616
377481,16,300534060218400,2023-07-10 03:20:00,-273332.045198,-14109.075176,C-SWOT,4.825800,38.549400,1.697002,38.380441,...,0.019985,-0.026271,-1.124046e-06,-1.032662e-06,0.0,1.0,SVP,0 days 08:23:38.131544384,577,2023-07-09 18:56:21.868455616
377482,16,300534060218400,2023-07-10 03:30:00,-273468.154017,-14293.282018,C-SWOT,4.825800,38.549400,1.695517,38.378741,...,0.019231,-0.026792,-1.387322e-06,-7.025868e-07,0.0,1.0,SVP,0 days 08:33:38.131544384,577,2023-07-09 18:56:21.868455616


In [5]:
#df = pd.read_csv(os.path.join(zarr_dir, 'drifters', f'drifterscoloc_'+colocs_sources.replace('.nc', '.csv')), dtype={'drifter_id':str}, parse_dates=['datetime'])
era515 = xr.open_dataset(os.path.join(zarr_dir, 'before_coloc', 'wind', 'era5_agesc_rio_z0.nc'))#.compute()
era50 = xr.open_dataset(os.path.join(zarr_dir, 'before_coloc', 'wind', 'era5_agesc_rio_z15.nc'))#.compute()
era5 = xr.merge([era50, era515]).sortby('latitude').compute()

# For small dataset (dt = '12h')

In [6]:
lon = df.longitude.values
lat = df.latitude.values
t = pd.to_datetime(df.datetime).values
ds0 = era5.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',t))
print('interp_ok')
df0 = ds0.to_dataframe()
df0.index.names = ['row_number']
df0.to_parquet(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+'.parquet'))

interp_ok


In [79]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v1.nc'
dt = '12h'
dfs = browse_swot_250m().reset_index()

def sel_drifters(dr, dt, swath, cycle):
    dfs_ = dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna()
    dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
    
    #time
    if dt == 'nearestswath':
        dr_ = dr.sel(datetime = slice(pd.to_datetime(dfs_.start_time_cut).values[0], pd.to_datetime(dfs_.end_time_cut).values[0]))
    else : 
        tmin = (pd.to_datetime(dfs_.time)-pd.Timedelta(dt)).values[0]
        tmax = (pd.to_datetime(dfs_.time)+pd.Timedelta(dt)).values[0]
        dr_ = dr.sel(datetime = slice(tmin, tmax))

    dr_= add_mask_inside_swot(dss, dr_)

    #under swath
    dr_ = dr_.where(dr_["inside_left"]+dr_["inside_right"])
    dfr_ = dr_.to_dataframe().reset_index().dropna()
    #time to nearest swot
    dfr_['time_to_swot']=(dfr_.datetime-dfs_.time.values[0]).abs()

    # stats for each drifters
    dfrs_=pd.DataFrame()
    dfrs_['time_to_swot_min'] = dfr_.groupby('drifter_id').time_to_swot.min()
    dfrs_['time_to_swot_max'] = dfr_.groupby('drifter_id').time_to_swot.max()
    dfrs_['point_number'] = dfr_.groupby('drifter_id').datetime.count()
    dfrs_['cycle_number'] = int(dfs_.cycle_number.values[0])
    dfrs_['cycle_date'] = dfs_.time.values[0]
    dfrs_['pass_number'] = int(dfs_.pass_number.values[0])
    return dfs_, dfr_, dfrs_.reset_index()

era5 = xr.open_dataset(os.path.join(zarr_dir, 'before_coloc', 'wind', 'era5_agesc_rio_spectral_decomp.nc'))
era5 = era5.where(era5.gap_mask==1)

D = []
for swath in [3,16]:
    for cycle in dfs.where(dfs.pass_number==swath).dropna().cycle_number:
        dfs_, dfr_, dfrs_ = sel_drifters(era5, dt, swath, cycle)
        dfr_['cycle_number'] = int(dfs_.cycle_number.values[0])
        dfr_['cycle_date'] = dfs_.time.values[0]
        dfr_['pass_number'] = int(dfs_.pass_number.values[0])
        D.append(dfr_)
        print(cycle)
        
dfw = pd.concat(D).set_index('pass_number').reset_index()
dfw = dfw.reset_index().sort_values(['drifter_id', 'datetime'])
dfw['row_number'] = df.reset_index().sort_values(['drifter_id', 'datetime']).row_number.values
dfw = dfw.sort_values('row_number').set_index('row_number').drop(columns=['index'])

# Store
dfw.to_csv(os.path.join(zarr_dir,'coloc_files', 'wind', f'windcoloc_spectral_decomp_{dt}_'+drifters_sources.replace('.nc', '.csv')))

474.0
475.0
476.0
478.0
479.0
480.0
481.0
482.0
483.0
484.0
485.0
486.0
487.0
488.0
489.0
490.0
491.0
492.0
493.0
494.0
495.0
496.0
497.0
499.0
500.0
501.0
502.0
503.0
504.0
505.0
506.0
507.0
508.0
509.0
510.0
476.0
478.0
479.0
481.0
482.0
483.0
484.0
485.0
486.0
487.0
488.0
489.0
490.0
491.0
492.0
493.0
494.0
495.0
496.0
497.0
498.0
499.0
500.0
501.0
502.0
503.0
504.0
505.0
506.0
507.0
509.0
510.0
511.0
512.0
514.0
515.0
516.0
517.0
518.0
519.0
520.0
521.0
522.0
523.0
524.0
525.0
529.0
530.0
531.0
532.0
533.0
535.0
536.0
537.0
538.0
539.0
540.0
541.0
542.0
543.0
544.0
545.0
546.0
547.0
548.0
549.0
550.0
551.0
552.0
553.0
555.0
556.0
557.0
558.0
559.0
560.0
561.0
562.0
563.0
564.0
565.0
566.0
567.0
569.0
570.0
571.0
572.0
573.0
574.0
575.0
576.0
577.0


In [81]:
df['datetime'] == dfw['datetime']

row_number
0         True
1         True
2         True
3         True
4         True
          ... 
377480    True
377481    True
377482    True
377483    True
377484    True
Name: datetime, Length: 377485, dtype: bool

In [93]:
list(dfw.columns)

['pass_number',
 'datetime',
 'drifter_id',
 'u10',
 'v10',
 't2m',
 'ewss',
 'iews',
 'inss',
 'msl',
 'nsss',
 'sst',
 'ssr',
 'ssrc',
 'str',
 'strc',
 'sp',
 'lgws',
 'zust',
 'lsm',
 'mwd',
 'tauoc',
 'mgws',
 'pp1d',
 'swh',
 'shts',
 'ust',
 'vst',
 'mwp',
 'f',
 'U10',
 'ue_agesc_z15',
 've_agesc_z15',
 'utauz_agesc_z15',
 'vtauz_agesc_z15',
 'us_agesc_z15',
 'vs_agesc_z15',
 'ues_agesc_z15',
 'ves_agesc_z15',
 'us0_agesc_z15',
 'vs0_agesc_z15',
 'ua_agesc_z15',
 'va_agesc_z15',
 'uw_agesc_z15',
 'vw_agesc_z15',
 'ustokes_agesc_z15',
 'vstokes_agesc_z15',
 'vsde_agesc_z15',
 'vsdn_agesc_z15',
 'ue_rioold_z15',
 've_rioold_z15',
 'ue_rio_z15',
 've_rio_z15',
 'vsde_rio_z15',
 'vsdn_rio_z15',
 'vsde_rioold_z15',
 'vsdn_rioold_z15',
 'Ue_agesc_z15',
 'Utauz_agesc_z15',
 'Us_agesc_z15',
 'Ues_agesc_z15',
 'Ue_rio_z15',
 'Ue_rioold_z15',
 'Ua_agesc_z15',
 'Uw_agesc_z15',
 'ue_agesc_z0',
 've_agesc_z0',
 'utauz_agesc_z0',
 'vtauz_agesc_z0',
 'us_agesc_z0',
 'vs_agesc_z0',
 'ues_agesc

______________
# For bigger dataset : Attempt of parallel not ok; alternative working but not beautiful

In [6]:
#MOCHE MAIS FONCTIONNE
def interp_one_part(df):

    lon = df.longitude.values
    lat = df.latitude.values
    time = df.datetime.values
    
    try :
        ds0 = era5.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',time))
    except :
        assert False, era5_
    df0 = ds0.to_dataframe()
    df0['row_number'] = df.reset_index()['row_number']
    return df0


for i in range(10):
    df_ = ddf.partitions[i].compute()
    d = interp_one_part(df_).set_index('row_number').to_csv(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+f'{i}.csv'))
    print(i)

0
1
2
3
4
5
6
7
8
9


In [7]:
files = sorted(glob(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+f'*.csv')))

In [8]:
files

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v10.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v11.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v12.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v13.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v14.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v15.csv',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_

In [9]:
D = []
for f in files :
    D.append(pd.read_csv(f).set_index('row_number'))
    print(f)
df = pd.concat(D, axis=0)
df.to_parquet(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+f'.parquet'))

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v10.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v11.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v12.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v13.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v14.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_v15.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC_v2.0.1/coloc_files/wind/era5_rioagesc_12h_spectral_decomp_all_med_variational_10min_

_______
# Data

In [ ]:
df = ddf.compute()

In [73]:
# Bins
time = era5.time.values
df['time_bin_min']= pd.cut(df['datetime'], time, labels=time[:-1])

In [7]:
#era5 = era5.chunk({'longitude':10, 'latitude':10, 'time':30}).persist()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 5.22 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


__________
# NOT WORKING : CSV coloc era5 (rio+agesc model)

In [8]:
def interp_one_group(df):
    dle = 0.25
    dt = pd.Timedelta('1h')
    lon = df.longitude.values
    lat = df.latitude.values
    time = df.datetime.values
    t = pd.to_datetime(df.time_bin_min.astype('datetime64[ns]').iloc[0])
    dt = pd.Timedelta('1h')
    
    era5_ = era5.sel(time = slice(t-dt/2,t+3/2*dt)).compute()
    
    try :
        ds0 = era5.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',time))
    except :
        assert False, era5_
    df0 = ds0.to_dataframe()
    df0.index.names = ['row_number']
    return df0


def interp_one_part(df):

    lon = df.longitude.values
    lat = df.latitude.values
    time = df.datetime.values
    
    try :
        ds0 = era5.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',time))
    except :
        assert False, era5_
    df0 = ds0.to_dataframe()
    df0.index.names = ['row_number']
    return df0

In [8]:
df_ = df[(df.datetime == df.time_bin_min.iloc[0]) ]
meta = interp_one_group(df_)
meta

NameError: name 'df' is not defined

In [12]:
dfout = ddf.groupby('time_bin_min').apply(interp_one_group, meta=meta).compute()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 1.46 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(

KeyboardInterrupt



In [10]:
df_ = ddf.partitions[0].compute()
meta = interp_one_group(df_)

In [11]:
meta

,u10,v10,t2m,ewss,iews,inss,msl,nsss,sst,ssr,...,Utauz_agesc_z0,Us_agesc_z0,Ues_agesc_z0,Ue_rio_z0,Ue_rioold_z0,Ua_agesc_z0,Uw_agesc_z0,longitude,latitude,time
row_number,,,,,,,,,,,,,,,,,,,,,
0,0.100759,0.357676,288.260363,0.109415,0.000054,0.000506,102273.043623,0.464068,288.378105,0.000000,...,0.000635,0.000259,0.003016,0.000564,0.000314,0.002237,0.002235,4.871798,40.779977,2023-04-10 20:00:00
1,0.135050,0.408442,288.246879,0.256505,0.000138,0.000574,102282.902676,0.735050,288.377857,0.000000,...,0.000624,0.000257,0.002986,0.000664,0.000369,0.002276,0.002216,4.870605,40.780433,2023-04-10 20:10:00
2,0.169215,0.459131,288.233386,0.403241,0.000222,0.000642,102292.765836,1.005647,288.377618,0.000000,...,0.000612,0.000254,0.002957,0.000762,0.000424,0.002315,0.002198,4.869407,40.780881,2023-04-10 20:20:00
3,0.203253,0.509745,288.219883,0.549603,0.000305,0.000709,102302.633112,1.275878,288.377387,0.000000,...,0.000601,0.000251,0.002927,0.000861,0.000478,0.002356,0.002180,4.868205,40.781321,2023-04-10 20:30:00
4,0.237160,0.560284,288.206372,0.695574,0.000389,0.000777,102312.504514,1.545763,288.377166,0.000000,...,0.000590,0.000248,0.002897,0.000959,0.000533,0.002397,0.002162,4.866998,40.781753,2023-04-10 20:40:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
753,2.099243,5.176215,288.824080,47.784090,0.015803,0.039259,101797.273080,122.257804,288.854618,52400.151794,...,NaN,NaN,NaN,0.045715,0.025397,NaN,NaN,4.470649,40.630309,2023-04-11 18:50:00
754,2.122940,5.255688,288.819318,52.366303,0.016271,0.040434,101795.546091,131.045768,288.854409,5533.227604,...,NaN,NaN,NaN,0.047081,0.026156,NaN,NaN,4.470035,40.630614,2023-04-11 19:00:00
755,2.157297,5.279177,288.821116,54.077634,0.016679,0.040865,101789.779245,133.963626,288.854201,4613.352110,...,NaN,NaN,NaN,0.047681,0.026490,NaN,NaN,4.469393,40.630923,2023-04-11 19:10:00


In [8]:
dfout = ddf.map_partitions(interp_one_part, meta=meta).compute()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 5.20 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/protocol/core.py", line 160, in loads
    return msgpack.loads(
  File "msgpack/_unpacker.pyx", line 194, in msgpack._cmsgpack.unpackb
ValueError: 3814429907 exceeds max_bin_len(2147483647)
distributed.core - ERROR - Exception while handling op register-client
Traceback (most recent call last):
  File "/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/core.py", line 968, in _handle_comm
    result = await result
  File "/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distribute

CancelledError: ('interp_one_part-dd473c20a7df880ee1b6ebe2b001474e', 6117)

In [7]:
#MOCHE MAIS FONCTIONNE
i=0
df_ = ddf.partitions[i].compute()
interp_one_part(df_).to_csv(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+'.csv'))


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39


In [8]:
dfw = dd.read_csv(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+'.csv')).set_index('row_number')

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/dataframe/io/csv.py:195: DtypeWarning: Columns (693) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/dataframe/io/csv.py:195: DtypeWarning: Columns (99) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/dataframe/io/csv.py:195: DtypeWarning: Columns (198,297,495,594,891,990) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/dataframe/io/csv.py:195: DtypeWarning: Columns (693) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)
/home1/datahome/mdemol/.miniconda

In [9]:
dfw

,u10,v10,t2m,ewss,iews,inss,msl,nsss,sst,ssr,ssrc,str,strc,sp,lgws,zust,lsm,mwd,tauoc,mgws,pp1d,swh,shts,ust,vst,mwp,f,ue_agesc_z15,ve_agesc_z15,utauz_agesc_z15,vtauz_agesc_z15,us_agesc_z15,vs_agesc_z15,ues_agesc_z15,ves_agesc_z15,us0_agesc_z15,vs0_agesc_z15,ua_agesc_z15,va_agesc_z15,uw_agesc_z15,vw_agesc_z15,ustokes_agesc_z15,vstokes_agesc_z15,vsde_agesc_z15,vsdn_agesc_z15,ue_rioold_z15,ve_rioold_z15,ue_rio_z15,ve_rio_z15,vsde_rio_z15,vsdn_rio_z15,vsde_rioold_z15,vsdn_rioold_z15,Ue_agesc_z15,Utauz_agesc_z15,Us_agesc_z15,Ues_agesc_z15,Ue_rio_z15,Ue_rioold_z15,Ua_agesc_z15,Uw_agesc_z15,U10,ue_agesc_z0,ve_agesc_z0,utauz_agesc_z0,vtauz_agesc_z0,us_agesc_z0,vs_agesc_z0,ues_agesc_z0,ves_agesc_z0,us0_agesc_z0,vs0_agesc_z0,ua_agesc_z0,va_agesc_z0,uw_agesc_z0,vw_agesc_z0,ustokes_agesc_z0,vstokes_agesc_z0,vsde_agesc_z0,vsdn_agesc_z0,ue_rioold_z0,ve_rioold_z0,ue_rio_z0,ve_rio_z0,vsde_rio_z0,vsdn_rio_z0,vsde_rioold_z0,vsdn_rioold_z0,Ue_agesc_z0,Utauz_agesc_z0,Us_agesc_z0,Ues_agesc_z0,Ue_rio_z0,Ue_rioold_z0,Ua_agesc_z0,Uw_agesc_z0,longitude,latitude,time,u10.1,v10.1,t2m.1,ewss.1,iews.1,inss.1,msl.1,nsss.1,sst.1,ssr.1,ssrc.1,str.1,strc.1,sp.1,lgws.1,zust.1,lsm.1,mwd.1,tauoc.1,mgws.1,pp1d.1,swh.1,shts.1,ust.1,vst.1,mwp.1,f.1,ue_agesc_z15.1,ve_agesc_z15.1,utauz_agesc_z15.1,vtauz_agesc_z15.1,us_agesc_z15.1,vs_agesc_z15.1,ues_agesc_z15.1,ves_agesc_z15.1,us0_agesc_z15.1,vs0_agesc_z15.1,ua_agesc_z15.1,va_agesc_z15.1,uw_agesc_z15.1,vw_agesc_z15.1,ustokes_agesc_z15.1,vstokes_agesc_z15.1,vsde_agesc_z15.1,vsdn_agesc_z15.1,ue_rioold_z15.1,ve_rioold_z15.1,ue_rio_z15.1,ve_rio_z15.1,vsde_rio_z15.1,vsdn_rio_z15.1,vsde_rioold_z15.1,vsdn_rioold_z15.1,Ue_agesc_z15.1,Utauz_agesc_z15.1,Us_agesc_z15.1,Ues_agesc_z15.1,Ue_rio_z15.1,Ue_rioold_z15.1,Ua_agesc_z15.1,Uw_agesc_z15.1,U10.1,ue_agesc_z0.1,ve_agesc_z0.1,utauz_agesc_z0.1,vtauz_agesc_z0.1,us_agesc_z0.1,vs_agesc_z0.1,ues_agesc_z0.1,ves_agesc_z0.1,us0_agesc_z0.1,vs0_agesc_z0.1,ua_agesc_z0.1,va_agesc_z0.1,uw_agesc_z0.1,vw_agesc_z0.1,ustokes_agesc_z0.1,vstokes_agesc_z0.1,vsde_agesc_z0.1,vsdn_agesc_z0.1,ue_rioold_z0.1,ve_rioold_z0.1,ue_rio_z0.1,ve_rio_z0.1,vsde_rio_z0.1,vsdn_rio_z0.1,vsde_rioold_z0.1,vsdn_rioold_z0.1,Ue_agesc_z0.1,Utauz_agesc_z0.1,Us_agesc_z0.1,Ues_agesc_z0.1,Ue_rio_z0.1,Ue_rioold_z0.1,Ua_agesc_z0.1,Uw_agesc_z0.1,longitude.1,latitude.1,time.1,u10.2,v10.2,t2m.2,ewss.2,iews.2,inss.2,msl.2,nsss.2,sst.2,ssr.2,ssrc.2,str.2,strc.2,sp.2,lgws.2,zust.2,lsm.2,mwd.2,tauoc.2,mgws.2,pp1d.2,swh.2,shts.2,ust.2,vst.2,mwp.2,f.2,ue_agesc_z15.2,ve_agesc_z15.2,utauz_agesc_z15.2,vtauz_agesc_z15.2,us_agesc_z15.2,vs_agesc_z15.2,ues_agesc_z15.2,ves_agesc_z15.2,us0_agesc_z15.2,vs0_agesc_z15.2,ua_agesc_z15.2,va_agesc_z15.2,uw_agesc_z15.2,vw_agesc_z15.2,ustokes_agesc_z15.2,vstokes_agesc_z15.2,vsde_agesc_z15.2,vsdn_agesc_z15.2,ue_rioold_z15.2,ve_rioold_z15.2,ue_rio_z15.2,ve_rio_z15.2,vsde_rio_z15.2,vsdn_rio_z15.2,vsde_rioold_z15.2,vsdn_rioold_z15.2,Ue_agesc_z15.2,Utauz_agesc_z15.2,Us_agesc_z15.2,Ues_agesc_z15.2,Ue_rio_z15.2,Ue_rioold_z15.2,Ua_agesc_z15.2,Uw_agesc_z15.2,U10.2,ue_agesc_z0.2,ve_agesc_z0.2,utauz_agesc_z0.2,vtauz_agesc_z0.2,us_agesc_z0.2,vs_agesc_z0.2,ues_agesc_z0.2,ves_agesc_z0.2,us0_agesc_z0.2,vs0_agesc_z0.2,ua_agesc_z0.2,va_agesc_z0.2,uw_agesc_z0.2,vw_agesc_z0.2,ustokes_agesc_z0.2,vstokes_agesc_z0.2,vsde_agesc_z0.2,vsdn_agesc_z0.2,ue_rioold_z0.2,ve_rioold_z0.2,ue_rio_z0.2,ve_rio_z0.2,vsde_rio_z0.2,vsdn_rio_z0.2,vsde_rioold_z0.2,vsdn_rioold_z0.2,Ue_agesc_z0.2,Utauz_agesc_z0.2,Us_agesc_z0.2,Ues_agesc_z0.2,Ue_rio_z0.2,Ue_rioold_z0.2,Ua_agesc_z0.2,Uw_agesc_z0.2,longitude.2,latitude.2,time.2,u10.3,v10.3,t2m.3,ewss.3,iews.3,inss.3,msl.3,nsss.3,sst.3,ssr.3,ssrc.3,str.3,strc.3,sp.3,lgws.3,zust.3,lsm.3,mwd.3,tauoc.3,mgws.3,pp1d.3,swh.3,shts.3,ust.3,vst.3,mwp.3,f.3,ue_agesc_z15.3,ve_agesc_z15.3,utauz_agesc_z15.3,vtauz_agesc_z15.3,us_agesc_z15.3,vs_agesc_z15.3,ues_agesc_z15.3,ves_agesc_z15.3,us0_agesc_z15.3,vs0_agesc_z15.3,ua_agesc_z15.3,va_agesc_z15.3,uw_agesc_z15.3,vw_agesc_z15.3,ustokes_agesc_z15.3,vstokes_agesc_z15

In [12]:
df =ddf.compute()


KeyboardInterrupt



In [6]:
df0.to_csv(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+'.csv'))


In [2]:
dsw = pd.read_csv(os.path.join(zarr_dir,'coloc_files','wind','era5_rioagesc_'+colocs_source.replace('.nc', '')+'.csv'))

NameError: name 'colocs_source' is not defined

In [ ]:
dsw

In [ ]:
def store_gaussian_250m(dsf, cutoff):
    
    def compute_gaussian_250m_one_file(da, cutoff):
        f = da['file'].values[0]
        from altimetry_filter import filter_diff_one
        method = 'gaussian'
        dg = filter_diff_one(f, filter_diff_method = method, filter_diff_kwargs = {'cutoff': cutoff})
        dg.to_netcdf(os.path.join(zarr_dir, 'before_coloc','preprocessed_swot','swot250m', f'{method}_{int(cutoff)}', f.split('/')[-1].replace('zarr', 'nc')))
        return da['file']
        
    template = dsf.file.isel(index = 0).compute()
    template = template.expand_dims({'index':dsf.index}).chunk({**dsf.chunks})
    
    d = dsf.map_blocks(compute_gaussian_250m_one_file, kwargs = {'cutoff':cutoff}, template=template).compute()
    return d